In [1]:
# n-ary windowed recording 

import random

# Windows signed representation parameter 
wSize  = 3
wDsize = wSize * 2
wMask  = (1 << wSize) - 1
wDMask = (1 << wDsize) - 1

# bls12_381_params
r = 0x73eda753299d7d483339d80809a1d80553bda402fffe5bfeffffffff00000001
Lamda =0xac45a4010001a40200000000ffffffff
w = 0x1a0111ea397fe699ec02408663d4de85aa0d857d89759ad4897d29650fb85f9b409427eb4f49fffd8bfd00000000aaac

In [2]:
# First Contribution : Unified scalar's recording/alignening

            # Naive GLV decomposition:
            # For a windows size of 3, the Table containe 32 elements of the form a * P + b * phi(P) such that a in {1,3,5,7} and b in {0,..,7}
            # Recording transforms the first coeficient to a negative Odd representation 

# Proof Of Concept of lookup table Generation
# The look up table here is generic, the goal of this code is just to illustrate the validiy of the recording scheme
# For Full and true generation of the lookup table Refere to the code in "Optimized-Multiplication.ipynb" 
Step = 2
LTv1 = [[1,0]]
LTv1 = LTv1 + [[LTv1[0][0] + Step, LTv1[0][1]]]
LTv1 = LTv1 + [[LTv1[1][0] + Step, LTv1[1][1]]]
LTv1 = LTv1 + [[LTv1[2][0] + Step, LTv1[2][1]]]
for j in range(28):LTv1 = LTv1 + [[LTv1[-4][0], LTv1[-4][1] + 1]]

# Unified proposed Signed-Digit Recording/Aligning Implementation
# Algorithm 3 from the paper ("Optimizing and securing GLV multiplication over BLS pairings-friendly curves").

def recordScalar_Algo3(scalar,lamda):
      # Scalar Decomposition
      x1 , x2 = scalar % lamda , scalar // lamda
      oddId = ((x1 & 1) - 1)
      x1    =  x1 - (oddId * lamda)           
      x2    = (x2 +  oddId)                 
      mu = (x1 | abs(x2)).bit_length()
      mu = mu + wDsize - (mu % wSize)
      # Scalar Recording 
      x1 = x1 | (1 << mu)   
      code =  1
      while (x1 != 1):
            sign = ((x1 >> wSize) & 1) - 1
            ai   = ((x1 ^ sign) |   1 ) & wMask 
            bi   = ((x2 + sign) ^ sign) & wMask
            code = (code << wDsize) | ((bi << wSize) + ai  + sign)
            sign = sign ^ ((~(bi + wMask) >> wSize) & sign)   
            x1   =  x1        >> wSize 
            x2   = (x2  >> wSize) + sign 
      return code >>3      

# Recover recorded digits from the final recorded representation of the scalar
# Input : Recording result
# Output : Set of recorded digits (ui, vi)
# Important! :This procedure is just used for demonstrative purposes: proof of concept of the decomposition validity 
def getDecompositionFromCode(code):
    u,v=[1],[code & wSize]
    code =code >> 3
    while (code !=1):
        sig =2*(code & 1) -1
        idx = (code & wDMask) >> 1      
        u=[sig*(LTv1[idx][0])] + u
        v=[sig*(LTv1[idx][1])] + v
        code = code >> wDsize
    return u,v

# Input decomposition from recording/aligning algortihm and validate its correctness with respect to the original scalar value
def verifyDecomposition(_u,_v, val):
            u,v=_u[-1],_v[-1]
            i=len(_u)-2
            while(i!=-1):
                  u = (u << wSize)+ _u[i]
                  v = (v << wSize)+ _v[i]
                  i=i-1
            return (u + v * Lamda) == val 

Scalar=(random.randint(0,r-1) ) # Random scalar 

sCode = recordScalar_Algo4(Scalar, Lamda)
Dec   = getDecompositionFromCode(sCode)
print("Scalar Value :",Scalar)
print("ui's values (recording of u) :", Dec[0])
print("vi's values (recording of v) :", Dec[1])
print("Scalar = VerififyRecording([ui],[vi]) :",verifyDecomposition(Dec[0], Dec[1], Scalar))

Scalar Value : 41839123405276480271161581763980297024931728208091214276493526980233011808460
ui's values (recording of u) : [1, 1, 5, 7, -1, -1, 7, -5, 5, -1, -1, 1, 3, 3, 7, 3, -3, -1, -5, -1, -5, -5, -3, -3, 3, -1, -1, 3, -3, -5, -5, 1, -5, -5, -3, -5, -3, 3, -5, -1, -5, 3, -5, 1]
vi's values (recording of v) : [5, 7, 1, 3, -5, -5, 6, 0, 6, -2, -2, 4, 6, 5, 7, 4, -3, -5, -3, -1, -5, -1, -2, -7, 3, -6, 0, 0, -7, -1, -7, 6, -3, -2, -4, -4, -5, 3, -3, -4, -6, 2, -6, 1]
Scalar = VerififyRecording([ui],[vi]) : True


In [3]:
# Second Contribution : Optimizing Lookup table computation
# Validation of the corresponding Recording/aligning for the new Table strucutre 

            # Modified GLV decomposition using Endomorphisme : Half of the table table is computed using endomorphisme
            # For a windows size of 3, the Table containe 32 elements of the form a * P + b * phi(P) such that a in {1,3,5,7} and b in {1,3,5,7}
            # Elements that have b Even are infered using the endomorphisme by the relation -phi(a * P + b * phi(P)) = b * P + (b - a) * phi(P)
            # since the relation generations negative values for (b - a), such entries in the table are modified during recording phase  
            # Recording transforms the first coeficient to a negative Odd representation 


# Proof Of Concept of Table Generation
# The look up table here is generic, the goal of this code is just to illustrate the validiy of the recording scheme
# For Full and true generation of the lookup table Refere to the code in "Optimized-Multiplication.ipynb" 
Step=2
LTv3=[[]]*32
LTv3[4] = [1,1]
LTv3[5] = [3,1]
LTv3[6] = [5,1]
LTv3[7] = [7,1]

for i in range(16):
      id1 = (i & 3) + ((i << 1) & (-8)) + 4
      id2 = ((id1 >> 3) + (((id1 >> 2) - ((id1 & 3) << 1) - 1) << 2)) % 32 
      if i > 3:
        LTv3[id1] = [LTv3[id1 - 8][0],LTv3[id1 - 8][1] + Step]
      LTv3[id2] = [LTv3[id1][1],LTv3[id1][1] - LTv3[id1][0]]  

# Proposed Signed-Digit Recording/Aligning Implementation for the new structure of the table computed using Endomorphisms
# Implementation of Algorithm 9 from the paper ("Optimizing and securing GLV multiplication over BLS pairings-friendly curves").

def recordScalar_Algo9(scalar,lamda):
    x1 , x2 = scalar % lamda , scalar // lamda
    oddId = ((x1 & 1) - 1)
    x1    =  x1 - (oddId * lamda)           
    x2    = (x2 +  oddId)                      
    mu = (x1 | abs(x2)).bit_length()
    mu = mu + wDsize - (mu % wSize)
    x1 = x1 | (1 << mu)   
    code =  1
    while (x1 != 1):
        sign = ((x1 >> wSize) & 1) - 1                 
        ai   = ((x1 ^ sign) |  1  ) & wMask 
        bi   = ((x2 + sign) ^ sign) & wMask
        code = (code << wDsize) | ((bi << wSize) + ai  + sign)
        inc  = (((x2 & wMask) - (x1 & wMask)) + wMask) >> wSize 
        x1   = (x1 >> wSize) 
        x2   = (x2 >> wSize) + inc - (inc + sign) * (x2 & 1) 
    return code >> 3

Scalar=(random.randint(0,r-1) | r)

# Recover recorded digits from the final recorded representation of the scalar
# Input : Recording result
# Output : Set of recorded digits (ui, vi)
# Important! :This procedure is just used for demonstrative purposes: proof of concept of the decomposition validity 
def getDecompositionFromCode(code):
    u,v = [1], [code & wSize]
    code = code >> 3
    while (code !=1):
        sig = (2*(code & 1)) - 1
        idx = (code & wDMask) >> 1 
        u = [sig * LTv3[idx][0]] + u
        v = [sig * LTv3[idx][1]] + v
        code = code >> wDsize
    return u,v
        
# Input decomposition from recording/aligning algortihm and validate its correctness with respect to the original scalar value
def verifyDecomposition(_u,_v, val):
            u,v = _u[-1], _v[-1]
            i = len(_u)-2
            while(i != -1):
                  u = (u << wSize) + _u[i]
                  v = (v << wSize) + _v[i]
                  i = i - 1
            return (u + v * Lamda) == val    

sCode = recordScalar_Algo9(Scalar, Lamda)
Dec   = getDecompositionFromCode(sCode)
print("Scalar Value :",Scalar)
print("ui's values (recording of u) :", Dec[0])
print("vi's values (recording of v) :", Dec[1])
print("Scalar = VerififyRecording([ui],[vi]) :",verifyDecomposition(Dec[0], Dec[1], Scalar))

Scalar Value : 54277482185927797070301633465703087985905179034368640842162840463073980811187
ui's values (recording of u) : [5, 1, 7, 5, -5, 3, -7, 3, 3, -7, 7, 3, -1, 1, -1, 3, -7, -5, 7, 3, 5, -5, 3, -7, -5, 3, -1, -5, -1, 5, -1, -7, -3, 7, -1, 3, -1, 1, 7, -5, -1, 1, -7, 1]
vi's values (recording of v) : [2, 3, 0, 2, -5, 0, -6, 3, 0, -3, 2, 1, -7, -6, -3, 1, -7, -4, 0, 0, 4, 0, -4, 0, -5, 3, 4, -4, -3, 5, 4, -2, 2, 7, 0, 5, 0, 5, 4, 2, 2, -2, -5, 1]
Scalar = VerififyRecording([ui],[vi]) : True


In [6]:
# Third Contribution : Space-Optimized lookup table:

            # Modified GLV decomposition using Endomorphisme :
            # For a windows size of 3, the Table containe only 16 elements of the form a * P + b * phi(P) such that a in {1,3,5,7} and b in {1,3,5,7}
            # Elements that have b Even are infered using the endomorphisme by the relation -phi(a * P + b * phi(P)) = b * P + (b - a) * phi(P)
            # since the relation generations negative values for (b - a), such entries in the table are modified during rcording phase  
            # Recording transforms the first coeficient to a negative Odd representation 

# Proof Of Concept of Table Generation
# The look up table here is generic, the goal of this code is just to illustrate the validiy of the recording scheme
# For Full and true generation of the lookup table Refere to the code in "Optimized-Multiplication.ipynb" 

Step = 2
LTv2 = [[1,1]]
LTv2 = LTv2 + [[LTv2[0][0] + Step, LTv2[0][1]]]
LTv2 = LTv2 + [[LTv2[1][0] + Step, LTv2[1][1]]]
LTv2 = LTv2 + [[LTv2[2][0] + Step, LTv2[2][1]]]
for j in range(12):LTv2 = LTv2 + [[LTv2[-4][0], LTv2[-4][1] + Step]]

# The following is a proof of Concept for digits generated using the proposed recording scheme for All possible combination when w=3.
# Each element of the Reference reperesent the followings:
#                              [(odd version of ui (1,3,5,7), vi), 
#                               sign (next limb is even->-1 else 1), 
#                               use fi (1 use "fi", 0 no), 
#                               indexes from table, 
#                               increment or not (1 increment, 0 no)]
# Coeficients are computed as follows:
#  if "fi" is used :
#           sign * (indexes[1]- indexes[0], indexes[1])
#  else :
#           sign * (indexes[0], indexes[1])       

Reference = [[[[(1,0),-1,1,(7,7),0],[(1,1),-1,0,(7,7),1],[(1,2),-1,1,(1,7),1],[(1,3),-1,0,(7,5),1],[(1,4),-1,1,(3,7),1],[(1,5),-1,0,(7,3),1],[(1,6),-1,1,(5,7),1],[(1,7),-1,0,(7,1),1]],
              [[(3,0),-1,1,(5,5),0],[(3,1),-1,0,(5,7),1],[(3,2),-1,1,(7,5),0],[(3,3),-1,0,(5,5),1],[(3,4),-1,1,(1,5),1],[(3,5),-1,0,(5,3),1],[(3,6),-1,1,(3,5),1],[(3,7),-1,0,(5,1),1]],
              [[(5,0),-1,1,(3,3),0],[(5,1),-1,0,(3,7),1],[(5,2),-1,1,(5,3),0],[(5,3),-1,0,(3,5),1],[(5,4),-1,1,(7,3),0],[(5,5),-1,0,(3,3),1],[(5,6),-1,1,(1,3),1],[(5,7),-1,0,(3,1),1]],
              [[(7,0),-1,1,(1,1),0],[(7,1),-1,0,(1,7),1],[(7,2),-1,1,(3,1),0],[(7,3),-1,0,(1,5),1],[(7,4),-1,1,(5,1),0],[(7,5),-1,0,(1,3),1],[(7,6),-1,1,(7,1),0],[(7,7),-1,0,(1,1),1]]],

             [[[(1,0),1,1,(1,1),0],[(1,1),1,0,(1,1),0],[(1,2),1,1,(7,1),1],[(1,3),1,0,(1,3),0],[(1,4),1,1,(5,1),1],[(1,5),1,0,(1,5),0],[(1,6),1,1,(3,1),1],[(1,7),1,0,(1,7),0]],  
              [[(3,0),1,1,(3,3),0],[(3,1),1,0,(3,1),0],[(3,2),1,1,(1,3),0],[(3,3),1,0,(3,3),0],[(3,4),1,1,(7,3),1],[(3,5),1,0,(3,5),0],[(3,6),1,1,(5,3),1],[(3,7),1,0,(3,7),0]],
              [[(5,0),1,1,(5,5),0],[(5,1),1,0,(5,1),0],[(5,2),1,1,(3,5),0],[(5,3),1,0,(5,3),0],[(5,4),1,1,(1,5),0],[(5,5),1,0,(5,5),0],[(5,6),1,1,(7,5),1],[(5,7),1,0,(5,7),0]],
              [[(7,0),1,1,(7,7),0],[(7,1),1,0,(7,1),0],[(7,2),1,1,(5,7),0],[(7,3),1,0,(7,3),0],[(7,4),1,1,(3,7),0],[(7,5),1,0,(7,5),0],[(7,6),1,1,(1,7),0],[(7,7),1,0,(7,7),0]]]]

Verified= True
for after in(0,8):
        for x1 in (1,3,5,7):
            for x2 in range(8):
               sign = ((after >> wSize) & 1) - 1                 # -1 if next is even else 0
               even = ~x2 & 1                                    # 1 if bi even
               ai   = ((x1 ^ sign) |  1  ) & wMask 
               bi   = ((x2 + sign) ^ sign) & wMask
               di   =  ai - bi
               inc  = ((wMask - (sign | 1) * di) & (bi + wMask)) >> wSize
               ai   = (ai - bi * even) & wMask
               bi   =  di * even + bi
               inc  =  inc * even + ((even - 1) * sign)  
               code = even + ((sign + 1) << 1) + (((ai >> 1) + ((bi - 1) << 1)) << 2)  # First bit: sign; Second bit: fi; Remaining bits : index in the table
               GeneratedCode = [(x1, x2),((code & 2) - 1), code & 1,(LTv2[code >> 2][0],LTv2[code >> 2][1]),inc]
               Verified = Verified & (GeneratedCode == Reference[sign + 1][x1 >> 1][x2])

print("Recording scheme verification with Respect to the Reference Structure :", Verified,"\n")

# Proposed Signed-Digit Recording/Aligning Implementation for the Space-optimized structure of the lookup table.
# Implementation of Algorithm 12 from the paper ("Optimizing and securing GLV multiplication over BLS pairings-friendly curves").
def recordScalar_Algo12(scalar,lamda):
    x1 , x2 = scalar % lamda , scalar // lamda
    oddId = ((x1 & 1) - 1)
    x1    =  x1 - (oddId * lamda)           
    x2    = (x2 +  oddId)                      
    mu = (x1 | abs(x2)).bit_length()
    mu = mu + wDsize - (mu % wSize)
    x1 = x1 | (1 << mu)   
    code =  1
    while (x1 != 1):
        sign = ((x1 >> wSize) & 1) - 1                 
        even = ~x2 & 1                                  
        ai   = ((x1 ^ sign) |  1  ) & wMask 
        bi   = ((x2 + sign) ^ sign) & wMask
        di   =  ai - bi
        inc  = ((wMask - (sign | 1) * di) & (bi + wMask)) >> wSize
        ai   = (ai - bi * even) & wMask
        bi   =  di * even + bi
        inc  =  inc * even + ((even - 1) * sign)  
        code = (code << wDsize) | (even + ((sign + 1) << 1) + (((ai >> 1) + ((bi - 1) << 1)) << 2))  
        x1   = (x1 >> wSize) 
        x2   = (x2 >> wSize) + inc
    return code

Scalar=(random.randint(0,r-1) | r)

# Recover recorded digits from the final recorded representation of the scalar
# Input : Recording result
# Output : Set of recorded digits (ui, vi)
# Important! :This procedure is just used for demonstrative purposes: proof of concept of the decomposition validity 
def getDecompositionFromCode(code):
    u,v = [], []
    while (code !=1):
        fi  =  code & 1
        sig = (code & 2) - 1
        idx = (code & wDMask) >> 2 
        if fi == 1:
            u = [sig * (LTv2[idx][1])] + u
            v = [sig * (LTv2[idx][1] - LTv2[idx][0])] + v
        else:
            u = [sig * (LTv2[idx][0])] + u
            v = [sig * (LTv2[idx][1])] + v
        code = code >> wDsize
    return u,v
        
# Input decomposition from recording/aligning algortihm and validate its correctness with respect to the original scalar value
def verifyDecomposition(_u,_v, val):
            u,v = _u[-1], _v[-1]
            i = len(_u)-2
            while(i != -1):
                  u = (u << wSize) + _u[i]
                  v = (v << wSize) + _v[i]
                  i = i - 1
            return (u + v * Lamda) == val    

sCode = recordScalar_Algo12(Scalar, Lamda)
Dec   = getDecompositionFromCode(sCode)
print("Scalar Value :",Scalar)
print("ui's values (recording of u) :", Dec[0])
print("vi's values (recording of v) :", Dec[1])
print("Scalar = VerififyRecording([ui],[vi]) :",verifyDecomposition(Dec[0], Dec[1], Scalar))

Recording scheme verification with Respect to the Reference Structure : True 

Scalar Value : 57892510910810586170735657496038536727744640606160501806876855591171469820867
ui's values (recording of u) : [7, 7, 5, 1, -5, 1, -3, 7, 5, -1, -3, -5, -3, 5, -1, 1, -5, 7, -3, -1, 3, -7, -5, 1, -1, 7, 1, -3, -1, 5, -1, -1, 3, 3, -7, 7, 5, -5, -5, -1, -3, 3, -3, -7, 1]
vi's values (recording of v) : [4, 7, -2, 0, -1, -6, -1, 4, 2, -7, 0, -7, -7, 5, -5, 1, -4, 6, -1, 4, 7, -4, -7, 5, 4, 6, 1, -3, 2, 1, -3, -5, 1, -4, -1, 7, 7, -3, -3, 2, -2, 0, -5, -7, 1]
Scalar = VerififyRecording([ui],[vi]) : True
